# Archetype Deep Dive — Notebook 4

In-depth analysis of each churn archetype:
- Profile statistics per archetype
- Feature distributions (Ghost vs others)
- Action effectiveness by archetype
- Archetype transition analysis
- Recommended campaign design per archetype

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.generate_data import generate_dataset
from src.features.feature_engineering import engineer_features, MODEL_FEATURES
from src.nba_engine.archetype_classifier import classify_archetypes, ARCHETYPES
from src.nba_engine.nba_engine import NBAEngine, add_revenue_impact
from src.clv.clv_calculator import add_clv_columns

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ARCHETYPE_COLORS = {'A': '#6c757d', 'B': '#dc3545', 'C': '#fd7e14', 'D': '#198754'}
ARCHETYPE_NAMES  = {'A': 'The Ghost', 'B': 'Frustrated Pro', 'C': 'Price Sensitive', 'D': 'Outgrown User'}
ARCHETYPE_EMOJI  = {'A': '👻', 'B': '😤', 'C': '💸', 'D': '🚀'}

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
print('Ready ✓')

## 1. Generate & Score Dataset

In [ ]:
df_raw = generate_dataset(n=8000)
df_eng = engineer_features(df_raw)

X = df_eng[MODEL_FEATURES].fillna(0).values
y = df_eng['churned'].values
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
scaler.fit(X_train)
X_s = scaler.transform(X)

model = lgb.LGBMClassifier(n_estimators=200, class_weight='balanced', random_state=42, verbose=-1)
model.fit(scaler.transform(X_train), y_train)

df_eng['churn_probability'] = model.predict_proba(X_s)[:, 1]
df_hr = df_eng[df_eng['churn_probability'] >= 0.50].copy().reset_index(drop=True)
df_hr = classify_archetypes(df_hr)
df_hr = add_clv_columns(df_hr)

engine = NBAEngine()
df_nba = engine.recommend_batch(df_hr)
df_nba = add_revenue_impact(df_nba)

print(f'High-risk customers: {len(df_hr):,}')
print(df_hr['churn_archetype'].value_counts())

## 2. Archetype Profile Cards

In [ ]:
profile_metrics = [
    'churn_probability', 'monthly_revenue', 'tenure_months',
    'engagement_decay_score', 'friction_score',
    'pricing_sensitivity_score', 'growth_pressure_score',
    'customer_health_score', 'clv',
]

profile = df_hr.groupby('churn_archetype')[profile_metrics].agg(['mean','median']).round(3)
print('Archetype Profile Statistics (mean | median):')
print(profile.to_string())

## 3. Archetype Score Radar Chart

In [ ]:
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

score_features = [
    'engagement_decay_score', 'friction_score',
    'pricing_sensitivity_score', 'growth_pressure_score',
    'retention_risk_score'
]
score_labels = ['Engagement\nDecay', 'Friction', 'Pricing\nSensitivity', 'Growth\nPressure', 'Retention\nRisk']

arch_means = df_hr.groupby('churn_archetype')[score_features].mean()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, arch in zip(axes, ['A','B','C','D']):
    vals = arch_means.loc[arch].values.tolist()
    x = np.arange(len(score_labels))
    color = ARCHETYPE_COLORS[arch]
    bars = ax.bar(x, vals, color=color, alpha=0.8, edgecolor='white', linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(score_labels, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'{ARCHETYPE_EMOJI[arch]} {ARCHETYPE_NAMES[arch]}', fontweight='bold', color=color)
    ax.set_ylabel('Score (0–1)' if arch == 'A' else '')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Archetype Diagnostic Signal Scores', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/nba/archetype_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Distribution by Archetype

In [ ]:
key_features = {
    'A': 'engagement_decay_score',
    'B': 'friction_score',
    'C': 'pricing_sensitivity_score',
    'D': 'growth_pressure_score',
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, (arch, feat) in zip(axes.flat, key_features.items()):
    for a, grp in df_hr.groupby('churn_archetype'):
        color = ARCHETYPE_COLORS[a]
        alpha = 0.85 if a == arch else 0.20
        lw = 2.0 if a == arch else 0.8
        grp[feat].plot.kde(ax=ax, color=color, alpha=alpha, linewidth=lw,
                           label=f'{ARCHETYPE_EMOJI[a]} {ARCHETYPE_NAMES[a]}')
    ax.set_title(f'Primary feature: {feat}\n(highlighted: {ARCHETYPE_EMOJI[arch]} {ARCHETYPE_NAMES[arch]})',
                 fontweight='bold', color=ARCHETYPE_COLORS[arch])
    ax.set_xlabel(feat)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions — Each Archetype vs Others', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/nba/archetype_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Revenue Opportunity by Archetype × CLV Tier

In [ ]:
heatmap_data = df_nba.groupby(['churn_archetype', 'clv_tier'])['expected_revenue_saved'].sum().unstack(fill_value=0)
tier_order = [t for t in ['Enterprise','High','Medium','Low'] if t in heatmap_data.columns]
heatmap_data = heatmap_data[tier_order]

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd',
    ax=ax, linewidths=0.5, linecolor='white',
    annot_kws={'size': 10},
)
ax.set_title('Revenue Opportunity ($) — Archetype × CLV Tier', fontsize=13, fontweight='bold')
ax.set_xlabel('CLV Tier')
ax.set_ylabel('Churn Archetype')
ax.set_yticklabels([f'{ARCHETYPE_EMOJI[a]} {a}' for a in heatmap_data.index], rotation=0)
plt.tight_layout()
plt.savefig('../reports/nba/revenue_heatmap_archetype_clv.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Action Distribution per Archetype

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, arch in zip(axes.flat, ['A','B','C','D']):
    grp = df_nba[df_nba['churn_archetype'] == arch]
    if len(grp) == 0:
        ax.set_visible(False)
        continue
    action_counts = grp['action'].value_counts()
    colors_pie = [ARCHETYPE_COLORS[arch]] * len(action_counts)
    # Vary shading
    shades = np.linspace(0.4, 0.95, len(action_counts))
    import matplotlib.colors as mcolors
    base_rgb = mcolors.to_rgb(ARCHETYPE_COLORS[arch])
    pie_colors = [tuple(s * c for c in base_rgb) for s in shades]

    ax.pie(action_counts.values, labels=None, colors=pie_colors,
           autopct='%1.0f%%', startangle=90, pctdistance=0.75,
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax.set_title(f'{ARCHETYPE_EMOJI[arch]} {ARCHETYPE_NAMES[arch]}\n(n={len(grp):,})',
                 fontweight='bold', color=ARCHETYPE_COLORS[arch])
    # Legend
    short_labels = [a[:35] + ('…' if len(a) > 35 else '') for a in action_counts.index]
    ax.legend(short_labels, loc='lower center', bbox_to_anchor=(0.5, -0.25),
              fontsize=7, ncol=1)

plt.suptitle('Recommended Actions Distribution by Archetype', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/nba/action_distribution_by_archetype.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Archetype Summary Table (Export-Ready)

In [ ]:
summary = df_nba.groupby('churn_archetype').agg(
    customers=('customer_id', 'count'),
    avg_churn_prob=('churn_probability', 'mean'),
    avg_mrr=('monthly_revenue', 'mean'),
    total_mrr=('monthly_revenue', 'sum'),
    avg_clv=('clv', 'mean'),
    total_rev_saved=('expected_revenue_saved', 'sum'),
    avg_conversion=('expected_conversion_rate', 'mean'),
    top_action=('action', lambda x: x.mode()[0]),
).round(2).reset_index()

summary['archetype_name'] = summary['churn_archetype'].map(ARCHETYPE_NAMES)
summary['avg_churn_prob'] = summary['avg_churn_prob'].map('{:.1%}'.format)
summary['avg_conversion'] = summary['avg_conversion'].map('{:.1%}'.format)
summary['avg_mrr'] = summary['avg_mrr'].map('${:,.0f}'.format)
summary['total_rev_saved'] = summary['total_rev_saved'].map('${:,.0f}'.format)
summary['avg_clv'] = summary['avg_clv'].map('${:,.0f}'.format)

display_cols = ['churn_archetype','archetype_name','customers','avg_churn_prob',
                'avg_mrr','avg_clv','total_rev_saved','avg_conversion','top_action']
print('\n=== ARCHETYPE EXECUTIVE SUMMARY ===')
print(summary[display_cols].to_string(index=False))

summary[display_cols].to_csv('../reports/nba/archetype_executive_summary.csv', index=False)
print('\nSaved to reports/nba/archetype_executive_summary.csv')